# LoRA Motion Style — Results Review
Experiment console: metrics, GIF gallery, A/B comparison, loss curves.

In [ ]:
import json
import os
from pathlib import Path
from IPython.display import Image, display, HTML
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

LOCAL_BASE = Path("/transfer/loraoutputs/eval")
MODEL_BASE = Path("/transfer/loraoutputs/models")

VERSIONS = {
    "v1": {"alpha": 16, "foot_vel": 0.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,  "note": "baseline"},
    "v2": {"alpha": 16, "foot_vel": 2.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,  "note": "+ foot velocity penalty"},
    "v3": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 0.0, "lr": 2e-4, "styles": 5,  "note": "+ lower alpha"},
    "v4": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 5,  "note": "+ root stability, lower lr"},
    "v5": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 20, "note": "20 styles"},
    "v6": {"alpha": 8,  "foot_vel": 2.0, "root_stable": 1.0, "lr": 1e-4, "styles": 21, "note": "21 styles (final)"},
}

STYLES_V6 = [
    "zombie", "elated", "old", "depressed", "drunk",
    "angry", "chicken", "proud",
    "heavyset", "bigsteps", "stiff", "duckfoot",
    "highknees", "flapping", "punch", "wildarms",
    "handsinpockets", "handsbetweenlegs", "onphoneleft",
    "penguin", "robot",
]

PROMPT_LABELS = [
    "a_person_walking_forward",
    "a_person_walking_in_a_circle",
    "a_person_stepping_sideways",
    "a_person_turning_around",
    "a_person_walking_and_then_stop",
]

PROMPT_NAMES = [
    "walking forward",
    "walking in circle",
    "stepping sideways",
    "turning around",
    "walk then stop",
]

def gif_path(version: str, style: str, prompt_idx: int = 0, kind: str = "cmp") -> Path:
    label = PROMPT_LABELS[prompt_idx]
    suffix = "" if version == "v1" else f"_{version}"
    viz_dir = LOCAL_BASE / f"multi_style{suffix}" / "viz"
    names = {"cmp": f"{prompt_idx}_cmp_{style}_{label}.gif",
             "base": f"{prompt_idx}_base_{label}.gif",
             "lora": f"{prompt_idx}_lora_{style}_{label}.gif"}
    return viz_dir / names.get(kind, names["cmp"])

print("Setup done.")

## 1. Experiment metadata

In [ ]:
pd.DataFrame(VERSIONS).T.rename_axis("version")

## 2. Quantitative results

In [ ]:
def load_eval(version: str) -> pd.DataFrame:
    suffix = "" if version == "v1" else f"_{version}"
    path = LOCAL_BASE / f"multi_style{suffix}" / "evaluation_results.json"
    if not path.exists():
        return pd.DataFrame()
    with open(path) as f:
        data = json.load(f)
    rows = [{"style": k.replace("lora_", "") if k != "base_model" else "base", **v}
            for k, v in data.items()]
    return pd.DataFrame(rows).set_index("style")

eval_data = {v: load_eval(v) for v in VERSIONS}

df = eval_data.get("v6", pd.DataFrame())
if not df.empty:
    display(df.sort_values("jitter_mean").style
            .background_gradient(subset=["diversity"], cmap="Blues")
            .background_gradient(subset=["jitter_mean"], cmap="Reds_r")
            .format("{:.2f}"))

In [ ]:
# Cross-version comparison for a single style
COMPARE_STYLE = "zombie"

rows = []
for v, df in eval_data.items():
    if not df.empty and COMPARE_STYLE in df.index:
        rows.append({"version": v, **df.loc[COMPARE_STYLE].to_dict()})
if rows:
    print(f"Cross-version: {COMPARE_STYLE}")
    display(pd.DataFrame(rows).set_index("version").style.format("{:.3f}"))

## 3. GIF gallery — one style across versions (walking forward + stepping sideways)

In [ ]:
GALLERY_STYLE    = "zombie"   # change as needed
GALLERY_VERSIONS = ["v1", "v3", "v5", "v6"]
GALLERY_PROMPTS  = [0, 2]     # walking forward + stepping sideways

for v in GALLERY_VERSIONS:
    print(f"\n── {v}: {VERSIONS[v]['note']}")
    for p in GALLERY_PROMPTS:
        path = gif_path(v, GALLERY_STYLE, p, "cmp")
        print(f"   {PROMPT_NAMES[p]}")
        if path.exists():
            display(Image(filename=str(path), width=700))
        else:
            print(f"   missing: {path.name}")

## 4. GIF gallery — v6, all styles (walking forward + stepping sideways)

In [ ]:
MULTI_VER   = "v6"
STYLES_SHOW = ["zombie", "drunk", "old", "chicken", "bigsteps", "duckfoot",
               "elated", "depressed", "angry", "wildarms", "robot", "penguin"]

for s in STYLES_SHOW:
    print(f"\n── {s}")
    for p in [0, 2]:   # walking forward, stepping sideways
        path = gif_path(MULTI_VER, s, p, "cmp")
        print(f"   {PROMPT_NAMES[p]}")
        if path.exists():
            display(Image(filename=str(path), width=700))
        else:
            print(f"   missing: {path.name}")

## 5. A/B comparison — pick style, version, prompt

In [ ]:
import base64

# Set these and run the cell
AB_STYLE   = "zombie"   # any style from STYLES_V6
AB_VERSION = "v6"       # v1 / v3 / v5 / v6

def _img_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

def _side_by_side(pairs):
    cols = []
    for path, label in pairs:
        if path.exists():
            b64 = _img_b64(path)
            cols.append(
                f'<div style="text-align:center;flex:1">'
                f'<b>{label}</b><br>'
                f'<img src="data:image/gif;base64,{b64}" style="width:100%">'
                f'</div>'
            )
        else:
            cols.append(
                f'<div style="text-align:center;flex:1">'
                f'<b>{label}</b><br>'
                f'<span style="color:#c00">missing: {path.name}</span>'
                f'</div>'
            )
    display(HTML(
        f'<div style="display:flex;gap:12px;margin-bottom:20px">'
        + "".join(cols) +
        f'</div>'
    ))

for p in [0, 2]:   # 0 = walking forward, 2 = stepping sideways
    print(f"── {PROMPT_NAMES[p]}")
    _side_by_side([
        (gif_path(AB_VERSION, AB_STYLE, p, "base"), "Base model"),
        (gif_path(AB_VERSION, AB_STYLE, p, "lora"), f"LoRA {AB_VERSION} — {AB_STYLE}"),
    ])

## 6. Metrics bar chart across versions

In [ ]:
CHART_STYLES   = ["zombie", "drunk", "old", "robot", "chicken", "bigsteps"]
CHART_METRIC   = "jitter_mean"   # or 'diversity'
CHART_VERSIONS = ["v1", "v3", "v5", "v6"]

rows = []
for v in CHART_VERSIONS:
    df = eval_data.get(v, pd.DataFrame())
    if df.empty:
        continue
    for s in CHART_STYLES:
        if s in df.index and CHART_METRIC in df.columns:
            rows.append({"version": v, "style": s, CHART_METRIC: df.loc[s, CHART_METRIC]})

if rows:
    pivot = pd.DataFrame(rows).pivot(index="style", columns="version", values=CHART_METRIC)
    ax = pivot.plot(kind="bar", figsize=(10, 4), width=0.7)
    ax.set_title(f"{CHART_METRIC} by style and version")
    ax.set_ylabel(CHART_METRIC)
    ax.set_xlabel("")
    ax.legend(title="version", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("No eval data. Check LOCAL_BASE path.")